In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Notebook folder:", Path.cwd())

Python: 3.13.1
pandas: 3.0.5
scikit-learn: 1.9.0
Notebook folder: C:\Users\yoana\Documents\PRT840-Thesis


In [3]:
# Update these folders if the dataset is stored elsewhere
dataset_folder = (
    Path.home()
    / "Downloads"
    / "CDU"
    / "PRT840 IT THESIS"
    / "CTU-SME-11_Experiment-VM-Microsoft-Windows7full-3_v1.0.0"
    / "CTU-SME-11"
    / "Experiment-VM-Microsoft-Windows7full-3"
)

Dataset folder exists: True
Connection logs found: 7
2023-02-20
2023-02-21
2023-02-22
2023-02-23
2023-02-24
2023-02-25
2023-02-26


In [4]:
def read_zeek_conn_log(log_path):
    fields = None

    with log_path.open(
        "r",
        encoding="utf-8",
        errors="replace"
    ) as log_file:
        for line in log_file:
            if line.startswith("#fields"):
                fields = line.rstrip().split("\t")[1:]
                break

    if fields is None:
        raise ValueError(
            f"No #fields header found in {log_path}"
        )

    day_data = pd.read_csv(
        log_path,
        sep="\t",
        comment="#",
        names=fields,
        dtype=str,
        na_values=["-", "(empty)"],
        low_memory=False
    )

    day_data["capture_date"] = (
        log_path.parents[1].name
    )

    return day_data


connection_tables = [
    read_zeek_conn_log(log_path)
    for log_path in conn_log_paths
]

combined_data = pd.concat(
    connection_tables,
    ignore_index=True
)

label_columns = [
    column
    for column in combined_data.columns
    if "label" in column.lower()
]

print(f"Days loaded: {len(connection_tables)}")
print(f"Total connections: {len(combined_data):,}")
print(f"Columns: {len(combined_data.columns)}")
print("Label columns:", label_columns)

Days loaded: 7
Total connections: 141,072
Columns: 24
Label columns: ['label', 'detailedlabel']


In [5]:
combined_data["label_clean"] = (
    combined_data["label"]
    .astype("string")
    .str.strip()
    .str.lower()
)

model_data = combined_data[
    combined_data["label_clean"].isin(
        ["benign", "malicious"]
    )
].copy()

model_data["target"] = (
    model_data["label_clean"]
    .map({
        "benign": 0,
        "malicious": 1
    })
    .astype(int)
)

print("All labels:")
print(combined_data["label_clean"].value_counts())

print(f"\nRecords used for binary ML: {len(model_data):,}")
print(
    "Unknown records held aside:",
    (combined_data["label_clean"] == "unknown").sum()
)

All labels:
label_clean
malicious    78971
benign       60794
unknown       1307
Name: count, dtype: Int64

Records used for binary ML: 139,765
Unknown records held aside: 1307


In [6]:
numeric_features = [
    "id.orig_p",
    "id.resp_p",
    "duration",
    "orig_bytes",
    "resp_bytes",
    "missed_bytes",
    "orig_pkts",
    "orig_ip_bytes",
    "resp_pkts",
    "resp_ip_bytes"
]

categorical_features = [
    "proto",
    "service",
    "conn_state",
    "history"
]

feature_columns = (
    numeric_features
    + categorical_features
)

missing_columns = [
    column
    for column in feature_columns
    if column not in model_data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

X = model_data[feature_columns].copy()
y = model_data["target"].copy()

for column in numeric_features:
    X[column] = pd.to_numeric(
        X[column],
        errors="coerce"
    )

for column in categorical_features:
    X[column] = (
        X[column]
        .fillna("missing")
        .astype(str)
    )

print("Input records:", f"{len(X):,}")
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Raw IP addresses included: No")
print("Target included as input: No")

Input records: 139,765
Numerical features: 10
Categorical features: 4
Raw IP addresses included: No
Target included as input: No


In [7]:
from sklearn.model_selection import train_test_split

train_indices, test_indices = train_test_split(
    X.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train = X.loc[train_indices].copy()
X_test = X.loc[test_indices].copy()
y_train = y.loc[train_indices].copy()
y_test = y.loc[test_indices].copy()

split_summary = pd.DataFrame({
    "training": y_train.value_counts(),
    "testing": y_test.value_counts()
}).rename(index={
    0: "benign",
    1: "malicious"
})

split_assignment = model_data[
    ["uid", "capture_date", "label_clean"]
].copy()

split_assignment["split"] = "training"
split_assignment.loc[test_indices, "split"] = "testing"

split_assignment.to_csv(
    model_output_folder
    / "decision_tree_split.csv",
    index_label="row_index"
)

print("Training records:", f"{len(X_train):,}")
print("Testing records:", f"{len(X_test):,}")
print()
print(split_summary)

Training records: 111,812
Testing records: 27,953

           training  testing
target                      
malicious     63177    15794
benign        48635    12159


In [8]:
from time import perf_counter

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

numeric_preprocessor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_preprocessor = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_preprocessor,
        numeric_features
    ),
    (
        "categorical",
        categorical_preprocessor,
        categorical_features
    )
])

baseline_decision_tree = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        DecisionTreeClassifier(
            random_state=42
        )
    )
])

start_time = perf_counter()

baseline_decision_tree.fit(
    X_train,
    y_train
)

training_seconds = perf_counter() - start_time

fitted_tree = baseline_decision_tree.named_steps[
    "classifier"
]

encoded_feature_count = len(
    baseline_decision_tree.named_steps[
        "preprocessor"
    ].get_feature_names_out()
)

print(
    "Decision Tree trained in:",
    f"{training_seconds:.2f} seconds"
)
print(
    "Features after encoding:",
    encoded_feature_count
)
print("Tree depth:", fitted_tree.get_depth())
print("Tree nodes:", fitted_tree.tree_.node_count)

Decision Tree trained in: 1.13 seconds
Features after encoding: 747
Tree depth: 19
Tree nodes: 65


In [9]:
from time import perf_counter

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        categorical_features
    )
])

baseline_decision_tree = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        DecisionTreeClassifier(
            random_state=42
        )
    )
])

start_time = perf_counter()

baseline_decision_tree.fit(
    X_train,
    y_train
)

training_seconds = perf_counter() - start_time

fitted_tree = baseline_decision_tree[
    "classifier"
]

encoded_feature_count = len(
    baseline_decision_tree[
        "preprocessor"
    ].get_feature_names_out()
)

print(
    "Decision Tree trained in:",
    f"{training_seconds:.2f} seconds"
)
print("Features after encoding:", encoded_feature_count)
print("Tree depth:", fitted_tree.get_depth())
print("Tree nodes:", fitted_tree.tree_.node_count)

Decision Tree trained in: 1.23 seconds
Features after encoding: 747
Tree depth: 19
Tree nodes: 65


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

train_predictions = (
    baseline_decision_tree.predict(X_train)
)

test_predictions = (
    baseline_decision_tree.predict(X_test)
)

def calculate_metrics(
    actual,
    predicted,
    dataset_name
):
    tn, fp, fn, tp = confusion_matrix(
        actual,
        predicted,
        labels=[0, 1]
    ).ravel()

    return {
        "dataset": dataset_name,
        "accuracy": accuracy_score(
            actual,
            predicted
        ),
        "precision": precision_score(
            actual,
            predicted,
            zero_division=0
        ),
        "recall": recall_score(
            actual,
            predicted,
            zero_division=0
        ),
        "f1_score": f1_score(
            actual,
            predicted,
            zero_division=0
        ),
        "false_positive_rate": (
            fp / (fp + tn)
            if (fp + tn) else 0
        )
    }

baseline_metrics = pd.DataFrame([
    calculate_metrics(
        y_train,
        train_predictions,
        "training"
    ),
    calculate_metrics(
        y_test,
        test_predictions,
        "testing"
    )
])

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1]
).ravel()

test_confusion_matrix = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp]
    ],
    index=[
        "actual_benign",
        "actual_malicious"
    ],
    columns=[
        "predicted_benign",
        "predicted_malicious"
    ]
)

baseline_metrics.to_csv(
    model_output_folder
    / "decision_tree_baseline_metrics.csv",
    index=False
)

test_confusion_matrix.to_csv(
    model_output_folder
    / "decision_tree_baseline_confusion_matrix.csv"
)

print("Baseline metrics:")
display(
    baseline_metrics.round(4)
)

print("Test confusion matrix:")
display(test_confusion_matrix)

Baseline metrics:


,dataset,accuracy,precision,recall,f1_score,false_positive_rate
0,training,1.0000,1.0,1.0000,1.0000,0.0
1,testing,0.9998,1.0,0.9996,0.9998,0.0


Test confusion matrix:


,predicted_benign,predicted_malicious
actual_benign,12159,0
actual_malicious,6,15788


In [11]:
feature_names = (
    baseline_decision_tree[
        "preprocessor"
    ].get_feature_names_out()
)

importance_table = pd.DataFrame({
    "feature": feature_names,
    "importance": baseline_decision_tree[
        "classifier"
    ].feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(
    importance_table
    .head(15)
    .round(4)
)

,feature,importance
716,categorical__history_Sr,0.9819
1,numeric__id.resp_p,0.0101
64,categorical__history_S,0.0064
7,numeric__orig_ip_bytes,0.0002
595,categorical__history_ShADTdtfF,0.0002
60,categorical__history_F,0.0002
4,numeric__resp_bytes,0.0001
580,categorical__history_ShADTdtaTr,0.0001
579,categorical__history_ShADTdtaTTtr,0.0001
587,categorical__history_ShADTdtar,0.0001


In [12]:
history_label_summary = pd.crosstab(
    model_data["history"].fillna("missing"),
    model_data["label_clean"]
)

history_label_summary["total"] = (
    history_label_summary.sum(axis=1)
)

history_label_summary[
    "malicious_percent"
] = (
    history_label_summary["malicious"]
    / history_label_summary["total"]
    * 100
).round(2)

print("The dominant Sr pattern:")
display(
    history_label_summary.loc[["Sr"]]
)

print("Most frequent history patterns:")
display(
    history_label_summary
    .sort_values(
        "total",
        ascending=False
    )
    .head(15)
)

The dominant Sr pattern:


label_clean,benign,malicious,total,malicious_percent
history,,,,
Sr,25,78356,78381,99.97


Most frequent history patterns:


label_clean,benign,malicious,total,malicious_percent
history,,,,
Sr,25,78356,78381,99.97
D,38306,0,38306,0.00
Dd,12137,6,12143,0.05
missing,4501,0,4501,0.00
S,768,554,1322,41.91
ShADTadtfF,502,0,502,0.00
ShADTadtFf,291,0,291,0.00
ShDdAFf,199,0,199,0.00
ShADTadtFfR,194,0,194,0.00


In [13]:
categorical_features_no_history = [
    "proto",
    "service",
    "conn_state"
]

features_no_history = (
    numeric_features
    + categorical_features_no_history
)

X_no_history = model_data[
    features_no_history
].copy()

for column in numeric_features:
    X_no_history[column] = pd.to_numeric(
        X_no_history[column],
        errors="coerce"
    )

for column in categorical_features_no_history:
    X_no_history[column] = (
        X_no_history[column]
        .fillna("missing")
        .astype(str)
    )

X_train_no_history = X_no_history.loc[
    train_indices
]
X_test_no_history = X_no_history.loc[
    test_indices
]

preprocessor_no_history = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        categorical_features_no_history
    )
])

decision_tree_no_history = Pipeline([
    (
        "preprocessor",
        preprocessor_no_history
    ),
    (
        "classifier",
        DecisionTreeClassifier(
            random_state=42
        )
    )
])

decision_tree_no_history.fit(
    X_train_no_history,
    y_train
)

no_history_predictions = (
    decision_tree_no_history.predict(
        X_test_no_history
    )
)

all_features_result = calculate_metrics(
    y_test,
    test_predictions,
    "All 14 features"
)

no_history_result = calculate_metrics(
    y_test,
    no_history_predictions,
    "Without history"
)

ablation_comparison = pd.DataFrame([
    all_features_result,
    no_history_result
]).rename(columns={
    "dataset": "feature_set"
})

ablation_comparison.to_csv(
    model_output_folder
    / "decision_tree_history_ablation.csv",
    index=False
)

display(
    ablation_comparison.round(4)
)

,feature_set,accuracy,precision,recall,f1_score,false_positive_rate
0,All 14 features,0.9998,1.0000,0.9996,0.9998,0.0000
1,Without history,0.9999,0.9999,0.9998,0.9999,0.0001


In [14]:
from sklearn.base import clone

model_dates = pd.to_datetime(
    model_data["capture_date"],
    errors="coerce"
)

temporal_train_mask = (
    model_dates
    <= pd.Timestamp("2023-02-21")
)

temporal_test_mask = (
    model_dates
    == pd.Timestamp("2023-02-22")
)

X_train_temporal = X.loc[
    temporal_train_mask
]
y_train_temporal = y.loc[
    temporal_train_mask
]

X_test_temporal = X.loc[
    temporal_test_mask
]
y_test_temporal = y.loc[
    temporal_test_mask
]

print("Temporal training labels:")
display(
    y_train_temporal
    .map({0: "benign", 1: "malicious"})
    .value_counts()
)

print("Temporal testing labels:")
display(
    y_test_temporal
    .map({0: "benign", 1: "malicious"})
    .value_counts()
)

temporal_decision_tree = clone(
    baseline_decision_tree
)

temporal_decision_tree.fit(
    X_train_temporal,
    y_train_temporal
)

temporal_train_predictions = (
    temporal_decision_tree.predict(
        X_train_temporal
    )
)

temporal_test_predictions = (
    temporal_decision_tree.predict(
        X_test_temporal
    )
)

temporal_metrics = pd.DataFrame([
    calculate_metrics(
        y_train_temporal,
        temporal_train_predictions,
        "20–21 February training"
    ),
    calculate_metrics(
        y_test_temporal,
        temporal_test_predictions,
        "22 February testing"
    )
])

temporal_metrics.to_csv(
    model_output_folder
    / "decision_tree_temporal_metrics.csv",
    index=False
)

display(
    temporal_metrics.round(4)
)

Temporal training labels:


target
malicious    44735
benign       16605
Name: count, dtype: int64

Temporal testing labels:


target
malicious    33742
benign        7187
Name: count, dtype: int64

,dataset,accuracy,precision,recall,f1_score,false_positive_rate
0,20–21 February training,1.0000,1.0,1.0000,1.0000,0.0000
1,22 February testing,0.9999,1.0,0.9999,0.9999,0.0001


In [15]:
temporal_tn, temporal_fp, temporal_fn, temporal_tp = (
    confusion_matrix(
        y_test_temporal,
        temporal_test_predictions,
        labels=[0, 1]
    ).ravel()
)

temporal_confusion_matrix = pd.DataFrame(
    [
        [temporal_tn, temporal_fp],
        [temporal_fn, temporal_tp]
    ],
    index=[
        "actual_benign",
        "actual_malicious"
    ],
    columns=[
        "predicted_benign",
        "predicted_malicious"
    ]
)

temporal_confusion_matrix.to_csv(
    model_output_folder
    / "decision_tree_temporal_confusion_matrix.csv"
)

display(temporal_confusion_matrix)

,predicted_benign,predicted_malicious
actual_benign,7186,1
actual_malicious,4,33738


In [16]:
from sklearn.base import clone
from sklearn.ensemble import IsolationForest

isolation_training_mask = (
    temporal_train_mask
    & (y == 0)
)

X_train_isolation = X.loc[
    isolation_training_mask
].copy()

X_test_isolation = (
    X_test_temporal.copy()
)

y_test_isolation = (
    y_test_temporal.copy()
)

baseline_isolation_forest = Pipeline([
    (
        "preprocessor",
        clone(preprocessor)
    ),
    (
        "detector",
        IsolationForest(
            n_estimators=200,
            contamination="auto",
            random_state=42,
            n_jobs=-1
        )
    )
])

start_time = perf_counter()

baseline_isolation_forest.fit(
    X_train_isolation
)

isolation_training_seconds = (
    perf_counter() - start_time
)

print(
    "Benign training records:",
    f"{len(X_train_isolation):,}"
)
print(
    "Testing records:",
    f"{len(X_test_isolation):,}"
)
print(
    "Isolation Forest trained in:",
    f"{isolation_training_seconds:.2f} seconds"
)
print(
    "Malicious labels used in training: No"
)

Benign training records: 16,605
Testing records: 40,929
Isolation Forest trained in: 0.51 seconds
Malicious labels used in training: No


In [17]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

raw_isolation_predictions = (
    baseline_isolation_forest.predict(
        X_test_isolation
    )
)

isolation_predictions = np.where(
    raw_isolation_predictions == -1,
    1,  # anomaly treated as malicious
    0   # normal treated as benign
)

isolation_anomaly_scores = (
    -baseline_isolation_forest
    .decision_function(
        X_test_isolation
    )
)

isolation_result = calculate_metrics(
    y_test_isolation,
    isolation_predictions,
    "22 February testing"
)

isolation_result["roc_auc"] = (
    roc_auc_score(
        y_test_isolation,
        isolation_anomaly_scores
    )
)

isolation_result[
    "average_precision"
] = average_precision_score(
    y_test_isolation,
    isolation_anomaly_scores
)

isolation_metrics = pd.DataFrame([
    isolation_result
])

isolation_tn, isolation_fp, isolation_fn, isolation_tp = (
    confusion_matrix(
        y_test_isolation,
        isolation_predictions,
        labels=[0, 1]
    ).ravel()
)

isolation_confusion_matrix = pd.DataFrame(
    [
        [isolation_tn, isolation_fp],
        [isolation_fn, isolation_tp]
    ],
    index=[
        "actual_benign",
        "actual_malicious"
    ],
    columns=[
        "predicted_benign",
        "predicted_malicious"
    ]
)

isolation_metrics.to_csv(
    model_output_folder
    / "isolation_forest_baseline_metrics.csv",
    index=False
)

isolation_confusion_matrix.to_csv(
    model_output_folder
    / "isolation_forest_baseline_confusion_matrix.csv"
)

print("Isolation Forest metrics:")
display(
    isolation_metrics.round(4)
)

print("Isolation Forest confusion matrix:")
display(isolation_confusion_matrix)

Isolation Forest metrics:


,dataset,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,average_precision
0,22 February testing,0.175,0.1471,0.0001,0.0003,0.004,0.746,0.8598


Isolation Forest confusion matrix:


,predicted_benign,predicted_malicious
actual_benign,7158,29
actual_malicious,33737,5


In [18]:
isolation_fit_mask = (
    (model_dates == pd.Timestamp("2023-02-20"))
    & (y == 0)
)

isolation_calibration_mask = (
    (model_dates == pd.Timestamp("2023-02-21"))
    & (y == 0)
)

X_isolation_fit = X.loc[
    isolation_fit_mask
]

X_isolation_calibration = X.loc[
    isolation_calibration_mask
]

calibrated_isolation_forest = Pipeline([
    (
        "preprocessor",
        clone(preprocessor)
    ),
    (
        "detector",
        IsolationForest(
            n_estimators=200,
            contamination="auto",
            random_state=42,
            n_jobs=-1
        )
    )
])

calibrated_isolation_forest.fit(
    X_isolation_fit
)

calibration_scores = (
    -calibrated_isolation_forest
    .decision_function(
        X_isolation_calibration
    )
)

one_percent_threshold = np.quantile(
    calibration_scores,
    0.99
)

calibrated_test_scores = (
    -calibrated_isolation_forest
    .decision_function(
        X_test_isolation
    )
)

calibrated_predictions = (
    calibrated_test_scores
    >= one_percent_threshold
).astype(int)

calibrated_result = calculate_metrics(
    y_test_isolation,
    calibrated_predictions,
    "1% benign-calibrated threshold"
)

calibrated_result["roc_auc"] = (
    roc_auc_score(
        y_test_isolation,
        calibrated_test_scores
    )
)

calibrated_result[
    "average_precision"
] = average_precision_score(
    y_test_isolation,
    calibrated_test_scores
)

calibrated_metrics = pd.DataFrame([
    calibrated_result
])

cal_tn, cal_fp, cal_fn, cal_tp = (
    confusion_matrix(
        y_test_isolation,
        calibrated_predictions,
        labels=[0, 1]
    ).ravel()
)

calibrated_confusion_matrix = pd.DataFrame(
    [
        [cal_tn, cal_fp],
        [cal_fn, cal_tp]
    ],
    index=[
        "actual_benign",
        "actual_malicious"
    ],
    columns=[
        "predicted_benign",
        "predicted_malicious"
    ]
)

calibrated_metrics.to_csv(
    model_output_folder
    / "isolation_forest_calibrated_metrics.csv",
    index=False
)

calibrated_confusion_matrix.to_csv(
    model_output_folder
    / "isolation_forest_calibrated_confusion_matrix.csv"
)

print(
    "Fit records:",
    f"{len(X_isolation_fit):,}"
)
print(
    "Calibration records:",
    f"{len(X_isolation_calibration):,}"
)
print(
    "Calibration threshold:",
    round(one_percent_threshold, 6)
)

display(
    calibrated_metrics.round(4)
)

display(calibrated_confusion_matrix)

Fit records: 7,837
Calibration records: 8,768
Calibration threshold: -0.01397


,dataset,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,average_precision
0,1% benign-calibrated threshold,0.1755,0.3333,0.0001,0.0003,0.0014,0.5992,0.8386


,predicted_benign,predicted_malicious
actual_benign,7177,10
actual_malicious,33737,5


In [19]:
numeric_only_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_features
    )
])

numeric_isolation_forest = Pipeline([
    (
        "preprocessor",
        numeric_only_preprocessor
    ),
    (
        "detector",
        IsolationForest(
            n_estimators=200,
            contamination="auto",
            random_state=42,
            n_jobs=-1
        )
    )
])

numeric_isolation_forest.fit(
    X_isolation_fit
)

numeric_calibration_scores = (
    -numeric_isolation_forest
    .decision_function(
        X_isolation_calibration
    )
)

numeric_threshold = np.quantile(
    numeric_calibration_scores,
    0.99
)

numeric_test_scores = (
    -numeric_isolation_forest
    .decision_function(
        X_test_isolation
    )
)

numeric_predictions = (
    numeric_test_scores
    >= numeric_threshold
).astype(int)

numeric_result = calculate_metrics(
    y_test_isolation,
    numeric_predictions,
    "10 numerical features"
)

numeric_result["roc_auc"] = (
    roc_auc_score(
        y_test_isolation,
        numeric_test_scores
    )
)

numeric_result[
    "average_precision"
] = average_precision_score(
    y_test_isolation,
    numeric_test_scores
)

numeric_comparison = pd.DataFrame([
    {
        **calibrated_result,
        "feature_set": "All 14 features"
    },
    {
        **numeric_result,
        "feature_set": "10 numerical features"
    }
])

num_tn, num_fp, num_fn, num_tp = (
    confusion_matrix(
        y_test_isolation,
        numeric_predictions,
        labels=[0, 1]
    ).ravel()
)

numeric_confusion_matrix = pd.DataFrame(
    [
        [num_tn, num_fp],
        [num_fn, num_tp]
    ],
    index=[
        "actual_benign",
        "actual_malicious"
    ],
    columns=[
        "predicted_benign",
        "predicted_malicious"
    ]
)

numeric_comparison.to_csv(
    model_output_folder
    / "isolation_forest_feature_ablation.csv",
    index=False
)

print(
    "Numerical-only threshold:",
    round(numeric_threshold, 6)
)

display(
    numeric_comparison[
        [
            "feature_set",
            "accuracy",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "average_precision"
        ]
    ].round(4)
)

display(numeric_confusion_matrix)

Numerical-only threshold: 0.224992


,feature_set,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,average_precision
0,All 14 features,0.1755,0.3333,0.0001,0.0003,0.0014,0.5992,0.8386
1,10 numerical features,0.1755,0.3125,0.0001,0.0003,0.0015,0.3299,0.7156


,predicted_benign,predicted_malicious
actual_benign,7176,11
actual_malicious,33737,5


In [20]:
def create_comparison_row(
    model_name,
    predictions,
    scores
):
    result = calculate_metrics(
        y_test_temporal,
        predictions,
        model_name
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test_temporal,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "model": model_name,
        "accuracy": result["accuracy"],
        "precision": result["precision"],
        "recall": result["recall"],
        "f1_score": result["f1_score"],
        "false_positive_rate": result[
            "false_positive_rate"
        ],
        "roc_auc": roc_auc_score(
            y_test_temporal,
            scores
        ),
        "average_precision": average_precision_score(
            y_test_temporal,
            scores
        ),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    }


decision_tree_scores = (
    temporal_decision_tree.predict_proba(
        X_test_temporal
    )[:, 1]
)

baseline_model_comparison = pd.DataFrame([
    create_comparison_row(
        "Decision Tree",
        temporal_test_predictions,
        decision_tree_scores
    ),
    create_comparison_row(
        "Isolation Forest - automatic threshold",
        isolation_predictions,
        isolation_anomaly_scores
    ),
    create_comparison_row(
        "Isolation Forest - 1% calibrated",
        calibrated_predictions,
        calibrated_test_scores
    ),
    create_comparison_row(
        "Isolation Forest - numerical only",
        numeric_predictions,
        numeric_test_scores
    )
])

baseline_model_comparison.to_csv(
    model_output_folder
    / "baseline_model_comparison.csv",
    index=False
)

display(
    baseline_model_comparison.round(4)
)

,model,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,average_precision,true_negatives,false_positives,false_negatives,true_positives
0,Decision Tree,0.9999,1.0000,0.9999,0.9999,0.0001,0.9999,0.9999,7186,1,4,33738
1,Isolation Forest - automatic threshold,0.1750,0.1471,0.0001,0.0003,0.0040,0.7460,0.8598,7158,29,33737,5
2,Isolation Forest - 1% calibrated,0.1755,0.3333,0.0001,0.0003,0.0014,0.5992,0.8386,7177,10,33737,5
3,Isolation Forest - numerical only,0.1755,0.3125,0.0001,0.0003,0.0015,0.3299,0.7156,7176,11,33737,5


In [21]:
rolling_daily_rows = []
all_rolling_actual = []
all_rolling_predictions = []

event_seconds = pd.to_numeric(
    model_data["ts"],
    errors="coerce"
)

rolling_test_dates = sorted(
    model_dates[
        model_dates > pd.Timestamp("2023-02-20")
    ].dropna().unique()
)

for test_date in rolling_test_dates:
    test_date = pd.Timestamp(test_date)

    prior_benign_indices = X.index[
        (model_dates < test_date)
        & (y == 0)
    ]

    prior_order = (
        event_seconds.loc[prior_benign_indices]
        .sort_values()
        .index
    )

    split_position = int(
        len(prior_order) * 0.80
    )

    rolling_fit_indices = prior_order[
        :split_position
    ]

    rolling_calibration_indices = prior_order[
        split_position:
    ]

    rolling_test_indices = X.index[
        model_dates == test_date
    ]

    rolling_model = Pipeline([
        (
            "preprocessor",
            clone(preprocessor)
        ),
        (
            "detector",
            IsolationForest(
                n_estimators=200,
                contamination="auto",
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    rolling_model.fit(
        X.loc[rolling_fit_indices]
    )

    rolling_calibration_scores = (
        -rolling_model.decision_function(
            X.loc[rolling_calibration_indices]
        )
    )

    rolling_threshold = np.quantile(
        rolling_calibration_scores,
        0.99
    )

    rolling_test_scores = (
        -rolling_model.decision_function(
            X.loc[rolling_test_indices]
        )
    )

    rolling_predictions = (
        rolling_test_scores
        >= rolling_threshold
    ).astype(int)

    rolling_actual = y.loc[
        rolling_test_indices
    ]

    daily_metrics = calculate_metrics(
        rolling_actual,
        rolling_predictions,
        str(test_date.date())
    )

    tn, fp, fn, tp = confusion_matrix(
        rolling_actual,
        rolling_predictions,
        labels=[0, 1]
    ).ravel()

    daily_metrics.update({
        "test_date": str(test_date.date()),
        "fit_records": len(
            rolling_fit_indices
        ),
        "calibration_records": len(
            rolling_calibration_indices
        ),
        "benign_test_records": int(
            (rolling_actual == 0).sum()
        ),
        "malicious_test_records": int(
            (rolling_actual == 1).sum()
        ),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "threshold": rolling_threshold
    })

    rolling_daily_rows.append(
        daily_metrics
    )

    all_rolling_actual.append(
        rolling_actual.to_numpy()
    )

    all_rolling_predictions.append(
        rolling_predictions
    )

    print(
        "Completed:",
        test_date.date()
    )


rolling_daily_results = pd.DataFrame(
    rolling_daily_rows
)

combined_actual = np.concatenate(
    all_rolling_actual
)

combined_predictions = np.concatenate(
    all_rolling_predictions
)

rolling_overall = pd.DataFrame([
    calculate_metrics(
        combined_actual,
        combined_predictions,
        "Combined 21–26 February"
    )
])

rolling_daily_results.to_csv(
    model_output_folder
    / "isolation_forest_rolling_daily_metrics.csv",
    index=False
)

rolling_overall.to_csv(
    model_output_folder
    / "isolation_forest_rolling_overall_metrics.csv",
    index=False
)

display(
    rolling_daily_results[
        [
            "test_date",
            "fit_records",
            "calibration_records",
            "benign_test_records",
            "malicious_test_records",
            "accuracy",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "false_positives",
            "false_negatives",
            "true_positives"
        ]
    ].round(4)
)

print("Combined rolling result:")
display(
    rolling_overall.round(4)
)

Completed: 2023-02-21
Completed: 2023-02-22
Completed: 2023-02-23
Completed: 2023-02-24
Completed: 2023-02-25
Completed: 2023-02-26


,test_date,fit_records,calibration_records,benign_test_records,malicious_test_records,accuracy,precision,recall,f1_score,false_positive_rate,false_positives,false_negatives,true_positives
0,2023-02-21,6269,1568,8768,44735,0.1522,0.0156,0.0002,0.0004,0.0722,633,44725,10
1,2023-02-22,13284,3321,7187,33742,0.1755,0.3333,0.0001,0.0003,0.0014,10,33737,5
2,2023-02-23,19033,4759,10478,459,0.9064,0.0154,0.0196,0.0173,0.0548,574,450,9
3,2023-02-24,27416,6854,13077,35,0.9973,0.5000,0.0571,0.1026,0.0002,2,33,2
4,2023-02-25,37877,9470,5846,0,0.9991,0.0000,0.0000,0.0000,0.0009,5,0,0
5,2023-02-26,42554,10639,7601,0,0.9779,0.0000,0.0000,0.0000,0.0221,168,0,0


Combined rolling result:


,dataset,accuracy,precision,recall,f1_score,false_positive_rate
0,Combined 21–26 February,0.3911,0.0183,0.0003,0.0006,0.0263


In [22]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

svm_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        categorical_features
    )
])

one_class_svm = Pipeline([
    (
        "preprocessor",
        svm_preprocessor
    ),
    (
        "detector",
        OneClassSVM(
            kernel="rbf",
            gamma="scale",
            nu=0.01
        )
    )
])

start_time = perf_counter()

one_class_svm.fit(
    X_isolation_fit
)

svm_training_seconds = (
    perf_counter() - start_time
)

svm_calibration_scores = (
    -one_class_svm.decision_function(
        X_isolation_calibration
    )
)

svm_threshold = np.quantile(
    svm_calibration_scores,
    0.99
)

svm_test_scores = (
    -one_class_svm.decision_function(
        X_test_isolation
    )
)

svm_predictions = (
    svm_test_scores
    >= svm_threshold
).astype(int)

svm_result = create_comparison_row(
    "One-Class SVM",
    svm_predictions,
    svm_test_scores
)

svm_metrics = pd.DataFrame([
    svm_result
])

svm_metrics.to_csv(
    model_output_folder
    / "one_class_svm_initial_metrics.csv",
    index=False
)

print(
    "Training time:",
    f"{svm_training_seconds:.2f} seconds"
)
print(
    "Support vectors:",
    len(
        one_class_svm[
            "detector"
        ].support_
    )
)
print(
    "Calibration threshold:",
    round(svm_threshold, 6)
)

display(
    svm_metrics.round(4)
)

Training time: 0.31 seconds
Support vectors: 115
Calibration threshold: 1.575564


,model,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,average_precision,true_negatives,false_positives,false_negatives,true_positives
0,One-Class SVM,0.1754,0.3,0.0002,0.0004,0.0019,0.0536,0.6403,7173,14,33736,6


In [23]:
svm_rolling_rows = []
svm_all_actual = []
svm_all_predictions = []

for test_date_value in rolling_test_dates:
    test_date = pd.Timestamp(
        test_date_value
    )

    prior_benign_indices = X.index[
        (model_dates < test_date)
        & (y == 0)
    ]

    prior_order = (
        event_seconds.loc[
            prior_benign_indices
        ]
        .sort_values()
        .index
    )

    split_position = int(
        len(prior_order) * 0.80
    )

    svm_fit_indices = prior_order[
        :split_position
    ]

    svm_calibration_indices = prior_order[
        split_position:
    ]

    svm_test_indices = X.index[
        model_dates == test_date
    ]

    rolling_svm = Pipeline([
        (
            "preprocessor",
            clone(svm_preprocessor)
        ),
        (
            "detector",
            OneClassSVM(
                kernel="rbf",
                gamma="scale",
                nu=0.01
            )
        )
    ])

    rolling_svm.fit(
        X.loc[svm_fit_indices]
    )

    calibration_scores = (
        -rolling_svm.decision_function(
            X.loc[
                svm_calibration_indices
            ]
        )
    )

    daily_threshold = np.quantile(
        calibration_scores,
        0.99
    )

    test_scores = (
        -rolling_svm.decision_function(
            X.loc[svm_test_indices]
        )
    )

    predictions = (
        test_scores
        >= daily_threshold
    ).astype(int)

    actual = y.loc[
        svm_test_indices
    ]

    daily_result = calculate_metrics(
        actual,
        predictions,
        str(test_date.date())
    )

    tn, fp, fn, tp = confusion_matrix(
        actual,
        predictions,
        labels=[0, 1]
    ).ravel()

    daily_result.update({
        "test_date": str(
            test_date.date()
        ),
        "fit_records": len(
            svm_fit_indices
        ),
        "calibration_records": len(
            svm_calibration_indices
        ),
        "benign_test_records": int(
            (actual == 0).sum()
        ),
        "malicious_test_records": int(
            (actual == 1).sum()
        ),
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    })

    if actual.nunique() == 2:
        daily_result["roc_auc"] = (
            roc_auc_score(
                actual,
                test_scores
            )
        )
    else:
        daily_result["roc_auc"] = np.nan

    svm_rolling_rows.append(
        daily_result
    )

    svm_all_actual.append(
        actual.to_numpy()
    )

    svm_all_predictions.append(
        predictions
    )

    print(
        "Completed:",
        test_date.date()
    )


svm_rolling_daily = pd.DataFrame(
    svm_rolling_rows
)

svm_combined_actual = np.concatenate(
    svm_all_actual
)

svm_combined_predictions = np.concatenate(
    svm_all_predictions
)

svm_rolling_overall = pd.DataFrame([
    calculate_metrics(
        svm_combined_actual,
        svm_combined_predictions,
        "Combined 21–26 February"
    )
])

svm_rolling_daily.to_csv(
    model_output_folder
    / "one_class_svm_rolling_daily_metrics.csv",
    index=False
)

svm_rolling_overall.to_csv(
    model_output_folder
    / "one_class_svm_rolling_overall_metrics.csv",
    index=False
)

display(
    svm_rolling_daily[
        [
            "test_date",
            "fit_records",
            "calibration_records",
            "benign_test_records",
            "malicious_test_records",
            "accuracy",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "false_positives",
            "false_negatives",
            "true_positives"
        ]
    ].round(4)
)

print("Combined rolling result:")
display(
    svm_rolling_overall.round(4)
)

Completed: 2023-02-21
Completed: 2023-02-22
Completed: 2023-02-23
Completed: 2023-02-24
Completed: 2023-02-25
Completed: 2023-02-26


,test_date,fit_records,calibration_records,benign_test_records,malicious_test_records,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,false_positives,false_negatives,true_positives
0,2023-02-21,6269,1568,8768,44735,0.1603,0.0563,0.0003,0.0005,0.0229,0.1160,201,44723,12
1,2023-02-22,13284,3321,7187,33742,0.1750,0.1622,0.0002,0.0004,0.0043,0.7479,31,33736,6
2,2023-02-23,19033,4759,10478,459,0.9297,0.0245,0.0174,0.0204,0.0303,0.6746,318,451,8
3,2023-02-24,27416,6854,13077,35,0.9971,0.3333,0.0857,0.1364,0.0005,0.7921,6,32,3
4,2023-02-25,37877,9470,5846,0,0.9827,0.0000,0.0000,0.0000,0.0173,NaN,101,0,0
5,2023-02-26,42554,10639,7601,0,0.9917,0.0000,0.0000,0.0000,0.0083,NaN,63,0,0


Combined rolling result:


,dataset,accuracy,precision,recall,f1_score,false_positive_rate
0,Combined 21–26 February,0.3962,0.0387,0.0004,0.0007,0.0136


In [24]:
decision_tree_daily_rows = []

later_test_dates = sorted(
    model_dates[
        model_dates
        >= pd.Timestamp("2023-02-22")
    ].dropna().unique()
)

for test_date_value in later_test_dates:
    test_date = pd.Timestamp(
        test_date_value
    )

    daily_indices = X.index[
        model_dates == test_date
    ]

    daily_actual = y.loc[
        daily_indices
    ]

    daily_predictions = (
        temporal_decision_tree.predict(
            X.loc[daily_indices]
        )
    )

    daily_scores = (
        temporal_decision_tree.predict_proba(
            X.loc[daily_indices]
        )[:, 1]
    )

    daily_result = calculate_metrics(
        daily_actual,
        daily_predictions,
        str(test_date.date())
    )

    tn, fp, fn, tp = confusion_matrix(
        daily_actual,
        daily_predictions,
        labels=[0, 1]
    ).ravel()

    daily_result.update({
        "test_date": str(
            test_date.date()
        ),
        "benign_test_records": int(
            (daily_actual == 0).sum()
        ),
        "malicious_test_records": int(
            (daily_actual == 1).sum()
        ),
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    })

    if daily_actual.nunique() == 2:
        daily_result["roc_auc"] = (
            roc_auc_score(
                daily_actual,
                daily_scores
            )
        )
    else:
        daily_result["roc_auc"] = np.nan

    decision_tree_daily_rows.append(
        daily_result
    )


decision_tree_daily_results = pd.DataFrame(
    decision_tree_daily_rows
)

later_test_mask = (
    model_dates
    >= pd.Timestamp("2023-02-22")
)

X_later_test = X.loc[
    later_test_mask
]

y_later_test = y.loc[
    later_test_mask
]

later_predictions = (
    temporal_decision_tree.predict(
        X_later_test
    )
)

later_scores = (
    temporal_decision_tree.predict_proba(
        X_later_test
    )[:, 1]
)

decision_tree_overall_result = (
    calculate_metrics(
        y_later_test,
        later_predictions,
        "Combined 22–26 February"
    )
)

decision_tree_overall_result[
    "roc_auc"
] = roc_auc_score(
    y_later_test,
    later_scores
)

decision_tree_overall_result[
    "average_precision"
] = average_precision_score(
    y_later_test,
    later_scores
)

overall_tn, overall_fp, overall_fn, overall_tp = (
    confusion_matrix(
        y_later_test,
        later_predictions,
        labels=[0, 1]
    ).ravel()
)

decision_tree_overall_result.update({
    "true_negatives": overall_tn,
    "false_positives": overall_fp,
    "false_negatives": overall_fn,
    "true_positives": overall_tp
})

decision_tree_overall = pd.DataFrame([
    decision_tree_overall_result
])

decision_tree_daily_results.to_csv(
    model_output_folder
    / "decision_tree_daily_temporal_metrics.csv",
    index=False
)

decision_tree_overall.to_csv(
    model_output_folder
    / "decision_tree_all_later_days_metrics.csv",
    index=False
)

display(
    decision_tree_daily_results[
        [
            "test_date",
            "benign_test_records",
            "malicious_test_records",
            "accuracy",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "false_positives",
            "false_negatives",
            "true_positives"
        ]
    ].round(4)
)

print("Combined Decision Tree result:")
display(
    decision_tree_overall.round(4)
)

,test_date,benign_test_records,malicious_test_records,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,false_positives,false_negatives,true_positives
0,2023-02-22,7187,33742,0.9999,1.0000,0.9999,0.9999,0.0001,0.9999,1,4,33738
1,2023-02-23,10478,459,0.9981,0.9888,0.9651,0.9768,0.0005,0.9823,5,16,443
2,2023-02-24,13077,35,0.9999,1.0000,0.9714,0.9855,0.0000,0.9857,0,1,34
3,2023-02-25,5846,0,1.0000,0.0000,0.0000,0.0000,0.0000,NaN,0,0,0
4,2023-02-26,7601,0,1.0000,0.0000,0.0000,0.0000,0.0000,NaN,0,0,0


Combined Decision Tree result:


,dataset,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,average_precision,true_negatives,false_positives,false_negatives,true_positives
0,Combined 22–26 February,0.9997,0.9998,0.9994,0.9996,0.0001,0.9996,0.9995,44183,6,21,34215


In [25]:
family_columns = [
    column
    for column in model_data.columns
    if "detailed" in column.lower()
]

print(
    "Attack-family columns:",
    family_columns
)

family_column = family_columns[0]

later_prediction_table = model_data.loc[
    later_test_mask,
    [
        "capture_date",
        "label_clean",
        "target",
        family_column
    ]
].copy()

later_prediction_table[
    "prediction"
] = later_predictions

later_prediction_table[
    "attack_family"
] = (
    later_prediction_table[
        family_column
    ]
    .fillna("unspecified")
    .astype(str)
)

malicious_family_performance = (
    later_prediction_table[
        later_prediction_table[
            "label_clean"
        ] == "malicious"
    ]
    .groupby(
        [
            "capture_date",
            "attack_family"
        ],
        as_index=False
    )
    .agg(
        malicious_records=(
            "target",
            "size"
        ),
        detected=(
            "prediction",
            "sum"
        )
    )
)

malicious_family_performance[
    "missed"
] = (
    malicious_family_performance[
        "malicious_records"
    ]
    - malicious_family_performance[
        "detected"
    ]
)

malicious_family_performance[
    "recall"
] = (
    malicious_family_performance[
        "detected"
    ]
    / malicious_family_performance[
        "malicious_records"
    ]
)

malicious_family_performance.to_csv(
    model_output_folder
    / "decision_tree_attack_family_performance.csv",
    index=False
)

display(
    malicious_family_performance.round(4)
)

Attack-family columns: ['detailedlabel']


,capture_date,attack_family,malicious_records,detected,missed,recall
0,2023-02-22,From_malicious-To_malicious-Malware_C2-Ingress...,3,3,0,1.0000
1,2023-02-22,From_malicious-To_malicious-Malware_data_exfil...,33739,33735,4,0.9999
2,2023-02-23,From_malicious-To_malicious-Human_attacks_data...,6,0,6,0.0000
3,2023-02-23,From_malicious-To_malicious-Malware_C2-Ingress...,11,6,5,0.5455
4,2023-02-23,From_malicious-To_malicious-Malware_data_exfil...,442,437,5,0.9887
5,2023-02-24,From_malicious-To_malicious-Malware_C2-Ingress...,3,3,0,1.0000
6,2023-02-24,From_malicious-To_malicious-Malware_data_exfil...,32,31,1,0.9688


In [26]:
family_overall_performance = (
    malicious_family_performance
    .groupby(
        "attack_family",
        as_index=False
    )
    .agg(
        malicious_records=(
            "malicious_records",
            "sum"
        ),
        detected=(
            "detected",
            "sum"
        ),
        missed=(
            "missed",
            "sum"
        )
    )
)

family_overall_performance[
    "recall"
] = (
    family_overall_performance[
        "detected"
    ]
    / family_overall_performance[
        "malicious_records"
    ]
)

macro_family_recall = (
    family_overall_performance[
        "recall"
    ].mean()
)

weighted_family_recall = (
    family_overall_performance[
        "detected"
    ].sum()
    / family_overall_performance[
        "malicious_records"
    ].sum()
)

family_overall_performance.to_csv(
    model_output_folder
    / "decision_tree_overall_attack_family_performance.csv",
    index=False
)

with pd.option_context(
    "display.max_colwidth",
    None
):
    display(
        family_overall_performance.round(4)
    )

print(
    "Macro-average family recall:",
    round(macro_family_recall, 4)
)

print(
    "Record-weighted recall:",
    round(weighted_family_recall, 4)
)

,attack_family,malicious_records,detected,missed,recall
0,From_malicious-To_malicious-Human_attacks_data_exfiltration-PyExfil-DNS,6,0,6,0.0000
1,From_malicious-To_malicious-Malware_C2-Ingress_tool_access-RemcosRAT-HTTP,17,12,5,0.7059
2,From_malicious-To_malicious-Malware_data_exfiltration-RemcosRAT,34213,34203,10,0.9997


Macro-average family recall: 0.5685
Record-weighted recall: 0.9994


In [27]:
common_prior_benign_indices = X.index[
    (model_dates < pd.Timestamp("2023-02-22"))
    & (y == 0)
]

common_prior_order = (
    event_seconds.loc[
        common_prior_benign_indices
    ]
    .sort_values()
    .index
)

common_split_position = int(
    len(common_prior_order) * 0.80
)

common_fit_indices = common_prior_order[
    :common_split_position
]

common_calibration_indices = common_prior_order[
    common_split_position:
]


common_isolation_forest = Pipeline([
    (
        "preprocessor",
        clone(preprocessor)
    ),
    (
        "detector",
        IsolationForest(
            n_estimators=200,
            contamination="auto",
            random_state=42,
            n_jobs=-1
        )
    )
])

common_isolation_forest.fit(
    X.loc[common_fit_indices]
)

common_if_calibration_scores = (
    -common_isolation_forest
    .decision_function(
        X.loc[
            common_calibration_indices
        ]
    )
)

common_if_threshold = np.quantile(
    common_if_calibration_scores,
    0.99
)

common_if_test_scores = (
    -common_isolation_forest
    .decision_function(
        X_later_test
    )
)

common_if_predictions = (
    common_if_test_scores
    >= common_if_threshold
).astype(int)


common_one_class_svm = Pipeline([
    (
        "preprocessor",
        clone(svm_preprocessor)
    ),
    (
        "detector",
        OneClassSVM(
            kernel="rbf",
            gamma="scale",
            nu=0.01
        )
    )
])

common_one_class_svm.fit(
    X.loc[common_fit_indices]
)

common_svm_calibration_scores = (
    -common_one_class_svm
    .decision_function(
        X.loc[
            common_calibration_indices
        ]
    )
)

common_svm_threshold = np.quantile(
    common_svm_calibration_scores,
    0.99
)

common_svm_test_scores = (
    -common_one_class_svm
    .decision_function(
        X_later_test
    )
)

common_svm_predictions = (
    common_svm_test_scores
    >= common_svm_threshold
).astype(int)


def common_comparison_row(
    model_name,
    predictions,
    scores
):
    result = calculate_metrics(
        y_later_test,
        predictions,
        model_name
    )

    tn, fp, fn, tp = confusion_matrix(
        y_later_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "model": model_name,
        "accuracy": result["accuracy"],
        "precision": result["precision"],
        "recall": result["recall"],
        "f1_score": result["f1_score"],
        "false_positive_rate": result[
            "false_positive_rate"
        ],
        "roc_auc": roc_auc_score(
            y_later_test,
            scores
        ),
        "average_precision": average_precision_score(
            y_later_test,
            scores
        ),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    }


common_baseline_comparison = pd.DataFrame([
    common_comparison_row(
        "Decision Tree",
        later_predictions,
        later_scores
    ),
    common_comparison_row(
        "Isolation Forest - 1% calibrated",
        common_if_predictions,
        common_if_test_scores
    ),
    common_comparison_row(
        "One-Class SVM - 1% calibrated",
        common_svm_predictions,
        common_svm_test_scores
    )
])

common_baseline_comparison.to_csv(
    model_output_folder
    / "common_test_baseline_comparison.csv",
    index=False
)

print(
    "Anomaly-model fit records:",
    len(common_fit_indices)
)

print(
    "Calibration records:",
    len(common_calibration_indices)
)

print(
    "Isolation Forest threshold:",
    round(common_if_threshold, 6)
)

print(
    "One-Class SVM threshold:",
    round(common_svm_threshold, 6)
)

display(
    common_baseline_comparison.round(4)
)

Anomaly-model fit records: 13284
Calibration records: 3321
Isolation Forest threshold: 0.016683
One-Class SVM threshold: 0.136364


,model,accuracy,precision,recall,f1_score,false_positive_rate,roc_auc,average_precision,true_negatives,false_positives,false_negatives,true_positives
0,Decision Tree,0.9997,0.9998,0.9994,0.9996,0.0001,0.9996,0.9995,44183,6,21,34215
1,Isolation Forest - 1% calibrated,0.5599,0.0485,0.0004,0.0009,0.0067,0.6134,0.4777,43895,294,34221,15
2,One-Class SVM - 1% calibrated,0.5598,0.0556,0.0005,0.0010,0.0069,0.7079,0.5250,43883,306,34218,18
